# Complete Improved Telco Recommendation Pipeline
## End-to-End: EDA → Preprocessing → Improved K-Means → FixedLightFM → XGBoost

**Data Source:** Raw CSV (`ac-01_telco_customer_behavior_mock_data.csv`)  
**Pipeline:** Complete preprocessing + improved modeling with performance optimizations

---

## Part 0: Setup & Imports

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Collaborative Filtering
import sys
import os
# Add backend to path (go up 2 levels from ml/notebook/ to project root, then to backend/)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(0, os.path.join(project_root, 'backend'))
from app.ml.models.collaborative.lightfm_recommender_fixed import FixedLightFMRecommender

# Ranking
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

# Utilities
from scipy.sparse import coo_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*80)
print("COMPLETE IMPROVED TELCO RECOMMENDATION PIPELINE")
print("="*80)

---
# PART 1: EXPLORATORY DATA ANALYSIS (EDA)
---

## 1.1 Load Raw Dataset

In [ ]:
# Load raw telco customer behavior data
path = '../data/raw/ac-01_telco_customer_behavior_mock_data.csv'
df_raw = pd.read_csv(path)

# Handle delimiter issues (CSV might have different delimiters)
if df_raw.shape[1] == 1:
    df_telco = df_raw.iloc[:, 0].str.split(',', expand=True)
    # Try semicolon if comma doesn't work
    if df_telco.shape[1] != 12:
        df_telco = df_raw.iloc[:, 0].str.split(';', expand=True)
else:
    df_telco = df_raw.copy()

# Set expected column names
expected_cols = [
    'customer_id', 'plan_type', 'device_brand', 'avg_data_usage_gb',
    'pct_video_usage', 'avg_call_duration', 'sms_freq', 'monthly_spend',
    'topup_freq', 'travel_score', 'complaint_count', 'target_offer'
]

if df_telco.shape[1] != len(expected_cols):
    raise ValueError(f"Expected {len(expected_cols)} columns, got {df_telco.shape[1]}")

df_telco.columns = expected_cols

# Convert numeric columns
num_cols = [
    'avg_data_usage_gb', 'pct_video_usage', 'avg_call_duration',
    'sms_freq', 'monthly_spend', 'topup_freq',
    'travel_score', 'complaint_count'
]
df_telco[num_cols] = df_telco[num_cols].apply(pd.to_numeric, errors='coerce')

print(f"✅ Loaded raw data: {df_telco.shape}")
print(f"\nColumns: {df_telco.columns.tolist()}")
print(f"\nFirst 5 rows:")
df_telco.head()

## 1.2 Data Overview & Quality Check

In [ ]:
print("📊 DATA OVERVIEW")
print("="*80)

print(f"\n1. Data Info:")
print(df_telco.info())

print(f"\n2. Missing Values:")
missing = df_telco.isnull().sum()
print(missing[missing > 0])

print(f"\n3. Basic Statistics:")
df_telco.describe()

## 1.3 Revenue Distribution Analysis

In [ ]:
# Revenue distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_telco['monthly_spend'], bins=50, edgecolor='black', color='coral')
axes[0].set_title('Monthly Spend Distribution', fontweight='bold')
axes[0].set_xlabel('Amount (IDR)')
axes[0].set_ylabel('Frequency')

# Box plot
axes[1].boxplot(df_telco['monthly_spend'])
axes[1].set_title('Monthly Spend Box Plot', fontweight='bold')
axes[1].set_ylabel('Amount (IDR)')

plt.tight_layout()
plt.show()

print(f"\n📊 Revenue Statistics:")
print(df_telco['monthly_spend'].describe())

## 1.4 User Behavior Analysis

In [ ]:
# Analyze key behavior metrics
user_behavior = df_telco[[
    'customer_id', 'monthly_spend', 'topup_freq',
    'avg_data_usage_gb', 'avg_call_duration',
    'sms_freq', 'complaint_count'
]].copy()

user_behavior.rename(columns={
    'monthly_spend': 'monetary',
    'topup_freq': 'frequency'
}, inplace=True)

print("📊 User Behavior Summary:")
print(user_behavior.describe())

# Visualize frequency and monetary distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_behavior['frequency'], bins=30, edgecolor='black', color='skyblue')
axes[0].set_title('Purchase Frequency Distribution', fontweight='bold')
axes[0].set_xlabel('Number of Top-ups')
axes[0].set_ylabel('Customers')

axes[1].hist(user_behavior['monetary'], bins=30, edgecolor='black', color='lightgreen')
axes[1].set_title('Total Spending Distribution', fontweight='bold')
axes[1].set_xlabel('Amount (IDR)')
axes[1].set_ylabel('Customers')

plt.tight_layout()
plt.show()

---
# PART 2: DATA PREPROCESSING
---

## 2.1 Handle Missing Values

In [ ]:
print("🔧 HANDLING MISSING VALUES")
print("="*80)

print(f"\nMissing values before:")
print(df_telco.isnull().sum())

# Categorical columns → fill with 'Unknown'
categorical_cols = ['plan_type', 'device_brand', 'target_offer']
for col in categorical_cols:
    df_telco[col] = df_telco[col].fillna('Unknown')

# Numeric columns → fill with median
numeric_cols = [
    'avg_data_usage_gb', 'pct_video_usage', 'avg_call_duration',
    'sms_freq', 'monthly_spend', 'topup_freq',
    'travel_score', 'complaint_count'
]
for col in numeric_cols:
    df_telco[col] = df_telco[col].fillna(df_telco[col].median())

print(f"\nMissing values after:")
print(df_telco.isnull().sum())
print("\n✅ Missing values handled")

## 2.2 Remove Duplicates & Filter Valid Data

In [ ]:
print("🔧 CLEANING DATA")
print("="*80)

# Remove duplicates
print(f"\nDuplicates before: {df_telco.duplicated().sum()}")
df_telco = df_telco.drop_duplicates()
print(f"Duplicates after: {df_telco.duplicated().sum()}")

# Filter valid customers
print(f"\nFiltering valid customers...")
df_telco = df_telco[
    (df_telco['monthly_spend'] > 0) &
    (df_telco['topup_freq'] > 0) &
    (df_telco['target_offer'].notna())
]

print(f"✅ Valid customers after filtering: {len(df_telco):,}")

## 2.3 Behavior Feature Engineering

In [ ]:
print("🔧 BEHAVIOR FEATURE ENGINEERING")
print("="*80)

# 1. Average spend per top-up
df_telco['avg_spend_per_topup'] = df_telco['monthly_spend'] / df_telco['topup_freq']

# 2. Data intensity (data usage vs spending)
df_telco['data_intensity'] = df_telco['avg_data_usage_gb'] / df_telco['monthly_spend']

# 3. Communication intensity (calls + SMS)
df_telco['communication_intensity'] = df_telco['avg_call_duration'] + df_telco['sms_freq']

# 4. Risk score
df_telco['risk_score'] = (
    df_telco['complaint_count'] * 0.7 + (1 - df_telco['travel_score']) * 0.3
)

print("✅ New features added:")
print(df_telco[[
    'avg_spend_per_topup', 'data_intensity',
    'communication_intensity', 'risk_score'
]].head())

## 2.4 RFM Features (Recency, Frequency, Monetary)

In [ ]:
print("🔧 RFM FEATURE CREATION")
print("="*80)

# Create RFM features
df_telco['recency'] = 1 / (df_telco['complaint_count'] + 1)  # Proxy for activity
df_telco['frequency'] = df_telco['topup_freq']
df_telco['monetary'] = df_telco['monthly_spend']

print("✅ RFM features created:")
print(df_telco[['customer_id', 'recency', 'frequency', 'monetary']].head())

# Visualize RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_telco['recency'], bins=50, edgecolor='black')
axes[0].set_title('Recency Distribution', fontweight='bold')
axes[0].set_xlabel('Recency Score')

axes[1].hist(df_telco['frequency'], bins=30, edgecolor='black')
axes[1].set_title('Frequency Distribution', fontweight='bold')
axes[1].set_xlabel('Top-up Frequency')

axes[2].hist(df_telco['monetary'], bins=50, edgecolor='black')
axes[2].set_title('Monetary Distribution', fontweight='bold')
axes[2].set_xlabel('Monthly Spend')

plt.tight_layout()
plt.show()

## 2.5 ARPU (Average Revenue Per User) Calculation

In [ ]:
print("🔧 ARPU CALCULATION")
print("="*80)

# Calculate ARPU
df_telco['arpu'] = df_telco['monthly_spend']  # Already monthly data

# ARPU buckets
df_telco['arpu_bucket'] = pd.cut(
    df_telco['arpu'],
    bins=[0, 50000, 100000, 200000, float('inf')],
    labels=['low', 'medium', 'high', 'premium']
)

print("✅ ARPU Bucket Distribution:")
print(df_telco['arpu_bucket'].value_counts())

# Visualize ARPU
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df_telco['arpu'], bins=50, edgecolor='black', color='teal')
axes[0].set_title('ARPU Distribution', fontweight='bold')
axes[0].set_xlabel('ARPU (IDR)')

counts = df_telco['arpu_bucket'].value_counts().sort_index()
axes[1].bar(range(len(counts)), counts.values, color='purple', edgecolor='black')
axes[1].set_title('ARPU Buckets', fontweight='bold')
axes[1].set_xlabel('Bucket')
axes[1].set_xticks(range(len(counts)))
axes[1].set_xticklabels(counts.index)

plt.tight_layout()
plt.show()

## 2.6 Usage Features (7-day proxy)

In [ ]:
print("🔧 USAGE FEATURES")
print("="*80)

# Create 7-day usage proxy features
np.random.seed(42)
df_telco['usage_7d_data_mb'] = (df_telco['avg_data_usage_gb'] * 1024 / 4).astype(int)
df_telco['usage_7d_voice_min'] = (df_telco['avg_call_duration'] * 7).astype(int)
df_telco['usage_7d_sms'] = (df_telco['sms_freq'] / 4).astype(int)

print("✅ Usage features created:")
print(df_telco[[
    'customer_id', 'usage_7d_data_mb',
    'usage_7d_voice_min', 'usage_7d_sms'
]].head())

## 2.7 Churn Score Calculation

In [ ]:
print("🔧 CHURN SCORE CALCULATION")
print("="*80)

# Normalize features to 0-1 scale
df_telco['data_norm'] = df_telco['avg_data_usage_gb'] / df_telco['avg_data_usage_gb'].max()
df_telco['call_norm'] = df_telco['avg_call_duration'] / df_telco['avg_call_duration'].max()
df_telco['spend_norm'] = df_telco['monthly_spend'] / df_telco['monthly_spend'].max()
df_telco['topup_norm'] = df_telco['topup_freq'] / df_telco['topup_freq'].max()
df_telco['complaint_norm'] = df_telco['complaint_count'] / df_telco['complaint_count'].max()

# Calculate churn score (0 = loyal, 1 = high risk)
df_telco['churn_score'] = (
    (1 - df_telco['data_norm']) * 0.25 +
    (1 - df_telco['call_norm']) * 0.20 +
    (1 - df_telco['spend_norm']) * 0.25 +
    (1 - df_telco['topup_norm']) * 0.15 +
    (df_telco['complaint_norm']) * 0.15
)

print("✅ Churn score created:")
print(df_telco[['customer_id', 'churn_score']].head())

# Visualize churn score distribution
plt.figure(figsize=(10, 5))
plt.hist(df_telco['churn_score'], bins=50, edgecolor='black', color='salmon')
plt.title('Churn Score Distribution', fontweight='bold')
plt.xlabel('Churn Score (0=Loyal, 1=High Risk)')
plt.ylabel('Customers')
plt.show()

## 2.8 Save Cleaned Data

In [ ]:
# Save cleaned telco data
df_telco.to_csv('telco_customers_cleaned.csv', index=False)
print("✅ Saved: telco_customers_cleaned.csv")

# Create user features dataset
user_features = df_telco[[
    'customer_id', 'recency', 'frequency', 'monetary',
    'arpu', 'arpu_bucket', 'usage_7d_data_mb',
    'usage_7d_voice_min', 'usage_7d_sms', 'churn_score'
]].copy()

user_features.to_csv('user_features.csv', index=False)
print("✅ Saved: user_features.csv")

print("\n" + "="*80)
print("✅ PREPROCESSING COMPLETE")
print("="*80)
print(f"Total Customers: {len(df_telco):,}")
print(f"Unique Customers: {df_telco['customer_id'].nunique():,}")
print(f"Unique Product Offers: {df_telco['target_offer'].nunique():,}")
print(f"Total Monthly Revenue: Rp {df_telco['monthly_spend'].sum():,.0f}")

---
# PART 3: IMPROVED K-MEANS SEGMENTATION
---

## 3.1 Improved Feature Selection

**Key Improvements:**
- Reduced from 6 to 4 decorrelated features
- Applied transformations to all skewed features
- Better feature correlation management

In [ ]:
print("\n" + "="*80)
print("PART 3: IMPROVED K-MEANS SEGMENTATION")
print("="*80)

print("\nSTEP 3.1: IMPROVED FEATURE SELECTION")
print("-"*80)

# Use only 4 most discriminative features
X_clustering = df_telco[['recency', 'frequency', 'monetary', 'churn_score']].copy()

# Apply transformations to skewed features
X_clustering['sqrt_recency'] = np.sqrt(X_clustering['recency'])
X_clustering['log_frequency'] = np.log1p(X_clustering['frequency'])
X_clustering['log_monetary'] = np.log1p(X_clustering['monetary'])

# Select transformed features
clustering_features = [
    'sqrt_recency',
    'log_frequency',
    'log_monetary',
    'churn_score'
]

X_cluster = X_clustering[clustering_features].fillna(0)

print(f"✅ Features selected: {clustering_features}")
print(f"\n✅ Feature correlation matrix:")
print(X_cluster.corr().round(3))

## 3.2 Robust Scaling

**Improvement:** Using RobustScaler instead of StandardScaler for better outlier handling

In [ ]:
print("\nSTEP 3.2: ROBUST SCALING")
print("-"*80)

# Use RobustScaler (better for outliers)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"✅ Applied RobustScaler (handles outliers better)")
print(f"   Scaled shape: {X_scaled.shape}")

## 3.3 Optimal K Selection

**Improvement:** Systematic K optimization using both silhouette and Davies-Bouldin scores

In [ ]:
print("\nSTEP 3.3: OPTIMAL K SELECTION")
print("-"*80)

k_range = range(2, 9)
silhouette_scores = []
db_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    
    sil_score = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    silhouette_scores.append(sil_score)
    db_scores.append(db_score)
    
    print(f"K={k}: Silhouette={sil_score:.4f}, Davies-Bouldin={db_score:.4f}")

# Find optimal K (maximize silhouette)
optimal_k_idx = np.argmax(silhouette_scores)
optimal_k = list(k_range)[optimal_k_idx]

print(f"\n✅ OPTIMAL K = {optimal_k} (Silhouette: {silhouette_scores[optimal_k_idx]:.4f})")

# Plot K selection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette plot
axes[0].plot(k_range, silhouette_scores, marker='o', linewidth=2)
axes[0].axvline(optimal_k, color='red', linestyle='--', label=f'Optimal K={optimal_k}')
axes[0].set_title('Silhouette Score vs K', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Silhouette Score')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Davies-Bouldin plot
axes[1].plot(k_range, db_scores, marker='o', linewidth=2, color='orange')
axes[1].set_title('Davies-Bouldin Index vs K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Davies-Bouldin Index (lower is better)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3.4 Train Improved K-Means

**Improvements:**
- Increased n_init (50 vs 10)
- Increased max_iter (500 vs 300)
- Using optimal K from analysis

In [ ]:
print("\nSTEP 3.4: TRAIN IMPROVED K-MEANS")
print("-"*80)

# Train with optimal K and improved parameters
kmeans_improved = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=50,  # More initializations
    max_iter=500  # More iterations
)

df_telco['segment_id'] = kmeans_improved.fit_predict(X_scaled)
user_features['segment_id'] = df_telco['segment_id']

# Evaluation
sil_improved = silhouette_score(X_scaled, df_telco['segment_id'])
db_improved = davies_bouldin_score(X_scaled, df_telco['segment_id'])

print(f"\n📊 IMPROVED K-MEANS RESULTS:")
print(f"   K                  : {optimal_k}")
print(f"   Silhouette Score   : {sil_improved:.4f}")
print(f"   Davies-Bouldin     : {db_improved:.4f}")
print(f"   Inertia            : {kmeans_improved.inertia_:.0f}")

print(f"\n🎯 COMPARISON:")
print(f"   Original Silhouette: 0.252")
print(f"   Improved Silhouette: {sil_improved:.4f}")
print(f"   Improvement: {((sil_improved - 0.252) / 0.252 * 100):+.1f}%")

print(f"\n✅ Segment Distribution:")
print(df_telco['segment_id'].value_counts().sort_index())

# Save improved segmentation
user_features.to_csv('user_features_improved_seg.csv', index=False)
print("\n✅ Saved: user_features_improved_seg.csv")

---
# PART 4: IMPROVED LIGHTFM COLLABORATIVE FILTERING
---

## 4.1 Create Multi-Transaction Data (CRITICAL FIX)

**Critical Fix:** Using actual product_id from target_offer (NOT segment_id!)
**Improvement:** Creating 2-5 transactions per user for realistic train/test split

In [ ]:
print("\n" + "="*80)
print("PART 4: IMPROVED LIGHTFM COLLABORATIVE FILTERING")
print("="*80)

print("\nSTEP 4.1: CREATE MULTI-TRANSACTION DATA (CRITICAL FIX)")
print("-"*80)

# Create multi-transaction data for realistic evaluation
np.random.seed(42)
transaction_list = []
base_date = pd.Timestamp('2024-01-01')

for idx, row in df_telco.iterrows():
    user_id = row['customer_id']
    n_transactions = np.random.randint(2, 6)  # 2-5 transactions per user
    
    for i in range(n_transactions):
        # Distribute transactions over 6 months
        days_offset = np.random.randint(0, 180)
        transaction_date = base_date + pd.Timedelta(days=days_offset)
        
        if i == 0:
            # First transaction: use target_offer (ground truth)
            product_id = row['target_offer']
            amount = row['monthly_spend']
        else:
            # Subsequent: random product with varied amount
            products = df_telco['target_offer'].unique()
            product_id = np.random.choice(products)
            amount = row['monthly_spend'] * np.random.uniform(0.7, 1.3)
        
        transaction_list.append({
            'user_id': user_id,
            'product_id': product_id,
            'amount': amount,
            'transaction_date': transaction_date
        })

transactions = pd.DataFrame(transaction_list)

print(f"✅ Created {len(transactions):,} transactions from {transactions['user_id'].nunique()} customers")
print(f"   Avg transactions per customer: {len(transactions) / transactions['user_id'].nunique():.1f}")
print(f"   Unique users: {transactions['user_id'].nunique()}")
print(f"   Unique products: {transactions['product_id'].nunique()}")
print(f"\n✅ Amount distribution:")
print(transactions['amount'].describe())

## 4.2 Chronological Train-Test Split

**Improvement:** Time-based split for realistic evaluation

In [ ]:
print("\nSTEP 4.2: CHRONOLOGICAL TRAIN-TEST SPLIT")
print("-"*80)

# Sort by transaction date
transactions = transactions.sort_values('transaction_date')
split_idx = int(len(transactions) * 0.8)

train_df = transactions.iloc[:split_idx].copy()
test_df = transactions.iloc[split_idx:].copy()

print(f"✅ Using chronological split by transaction_date")
print(f"Train: {len(train_df):,} transactions")
print(f"  Users: {train_df['user_id'].nunique()}")
print(f"  Products: {train_df['product_id'].nunique()}")
print(f"Test: {len(test_df):,} transactions")
print(f"  Users: {test_df['user_id'].nunique()}")
print(f"  Products: {test_df['product_id'].nunique()}")

## 4.3 Initialize FixedLightFM Model

**Improvements:**
- Increased components (128 vs 50)
- WARP loss for ranking optimization
- Proper hyperparameters

In [ ]:
print("\nSTEP 4.3: INITIALIZE FIXEDLIGHTFM MODEL")
print("-"*80)

# Initialize with improved hyperparameters
lightfm_improved = FixedLightFMRecommender(
    no_components=128,  # Increased from 50
    loss='warp',
    learning_rate=0.05,
    user_alpha=1e-6,
    item_alpha=1e-6
)

print(f"✅ FixedLightFM initialized:")
print(f"   Components: 128")
print(f"   Loss: WARP (ranking optimization)")
print(f"   Learning rate: 0.05")

## 4.4 Prepare Interaction Matrices

**Improvement:** Weighted ratings based on transaction amounts (0.5-1.0 range)

In [ ]:
print("\nSTEP 4.4: PREPARE INTERACTION MATRIX")
print("-"*80)

# Prepare train interactions with weighted ratings
train_interactions = lightfm_improved.prepare_data(
    interactions_df=train_df[['user_id', 'product_id', 'amount']],
    weight_by_amount=True
)

print(f"✅ Train interaction matrix prepared:")
print(f"   Shape: {train_interactions.shape}")
print(f"   Users: {train_interactions.shape[0]}")
print(f"   Products: {train_interactions.shape[1]}")
print(f"   Non-zero interactions: {train_interactions.nnz:,}")
print(f"   Sparsity: {1 - train_interactions.nnz / (train_interactions.shape[0] * train_interactions.shape[1]):.4%}")

# Prepare test interactions
test_user_indices = test_df['user_id'].map(lightfm_improved.user_id_map)
test_product_indices = test_df['product_id'].map(lightfm_improved.product_id_map)

# Filter cold start samples
test_mask = test_user_indices.notna() & test_product_indices.notna()
test_filtered = test_df[test_mask].copy()

print(f"\n✅ Test data filtered:")
print(f"   Original: {len(test_df)} transactions")
print(f"   Filtered: {len(test_filtered)} transactions")
print(f"   Cold start removed: {len(test_df) - len(test_filtered)}")

# Create test interaction matrix
if len(test_filtered) > 0:
    test_user_idx = test_filtered['user_id'].map(lightfm_improved.user_id_map).values
    test_product_idx = test_filtered['product_id'].map(lightfm_improved.product_id_map).values
    
    # Weighted ratings
    amounts = test_filtered['amount'].values
    min_amount = amounts.min()
    max_amount = amounts.max()
    if max_amount > min_amount:
        test_ratings = 0.5 + 0.5 * (amounts - min_amount) / (max_amount - min_amount)
    else:
        test_ratings = np.ones(len(amounts))
    
    test_interactions = coo_matrix(
        (test_ratings, (test_user_idx, test_product_idx)),
        shape=train_interactions.shape,
        dtype=np.float32
    )
    
    print(f"✅ Test interaction matrix: {test_interactions.shape}, nnz: {test_interactions.nnz}")
else:
    print(f"⚠️ No test data, using train as test")
    test_interactions = train_interactions

## 4.5 Train Improved FixedLightFM

**Improvement:** 100 epochs (vs 30) for better convergence

In [ ]:
print("\nSTEP 4.5: TRAIN IMPROVED FIXEDLIGHTFM")
print("-"*80)

print("Training FixedLightFM (100 epochs)...")
train_metrics = lightfm_improved.train(
    interaction_matrix=train_interactions,
    epochs=100,
    num_threads=4,
    verbose=True
)

print(f"\n✅ Training complete!")
print(f"\n📊 Training metrics:")
for metric_name, value in train_metrics.items():
    print(f"   {metric_name}: {value:.4f}")

## 4.6 Evaluate Improved FixedLightFM

In [ ]:
print("\nSTEP 4.6: EVALUATE IMPROVED FIXEDLIGHTFM")
print("-"*80)

# Evaluate on test set
test_metrics = lightfm_improved.evaluate(
    test_interactions,
    k_values=[5, 10, 20]
)

print(f"\n📊 IMPROVED FIXEDLIGHTFM TEST RESULTS:")
print(f"   Precision@5   : {test_metrics.get('test_precision_at_5', 0):.4f}")
print(f"   Precision@10  : {test_metrics.get('test_precision_at_10', 0):.4f}")
print(f"   Precision@20  : {test_metrics.get('test_precision_at_20', 0):.4f}")
print(f"   Recall@5      : {test_metrics.get('test_recall_at_5', 0):.4f}")
print(f"   Recall@10     : {test_metrics.get('test_recall_at_10', 0):.4f}")
print(f"   Recall@20     : {test_metrics.get('test_recall_at_20', 0):.4f}")
print(f"   AUC           : {test_metrics.get('test_auc', 0):.4f}")

test_precision = test_metrics.get('test_precision_at_5', 0)
test_recall = test_metrics.get('test_recall_at_10', 0)
test_auc = test_metrics.get('test_auc', 0)

print(f"\n🎯 COMPARISON vs OLD LIGHTFM:")
print(f"   Original Precision@5: 0.200")
print(f"   Improved Precision@5: {test_precision:.4f}")
print(f"   Improvement: {((test_precision - 0.200) / 0.200 * 100):+.1f}%")

print(f"\n🎯 TARGET ACHIEVEMENT:")
print(f"   Target Precision@5 ≥ 0.70: {test_precision:.4f} {'✅ PASS' if test_precision >= 0.70 else '❌ FAIL'}")
print(f"   Target Recall@10 ≥ 0.75:   {test_recall:.4f} {'✅ PASS' if test_recall >= 0.75 else '❌ FAIL'}")
print(f"   Target AUC ≥ 0.85:         {test_auc:.4f} {'✅ PASS' if test_auc >= 0.85 else '❌ FAIL'}")

---
# PART 5: IMPROVED XGBOOST RANKING
---

## 5.1 Generate Improved Candidates

**Fix:** Using train users (known to model) for candidate generation

In [ ]:
print("\n" + "="*80)
print("PART 5: IMPROVED XGBOOST RANKING")
print("="*80)

print("\nSTEP 5.1: GENERATE IMPROVED CANDIDATES")
print("-"*80)

# Use train users who also appear in test for evaluation
train_user_set = set(train_df['user_id'].unique())
test_user_set = set(test_df['user_id'].unique()) if len(test_df) > 0 else set()

sample_users = list(train_user_set.intersection(test_user_set))[:1500]

if len(sample_users) == 0:
    print("⚠️ No overlap between train/test users, using train users only")
    sample_users = list(train_df['user_id'].unique())[:1500]

print(f"ℹ️  Sampling from {len(sample_users)} users (known to FixedLightFM model)")

candidates_improved = {}

for user_id in sample_users:
    try:
        recommendations = lightfm_improved.recommend(
            user_id=user_id,
            n_recommendations=50
        )
        
        candidates_improved[user_id] = [
            (rec['product_id'], rec['score'])
            for rec in recommendations
        ]
    except Exception as e:
        candidates_improved[user_id] = []

avg_candidates = np.mean([len(c) for c in candidates_improved.values()]) if len(candidates_improved) > 0 else 0
print(f"✅ Generated candidates for {len(candidates_improved)} users")
print(f"   Avg candidates per user: {avg_candidates:.1f}")

## 5.2 Build Improved Ranking Dataset

**Improvements:**
- Added 4 new interaction features
- Better feature engineering (freq × monetary, arpu/usage, loyalty score)

In [ ]:
print("\nSTEP 5.2: BUILD IMPROVED RANKING DATASET")
print("-"*80)

xgb_improved = None
ranking_data_improved = []

for user_id in sample_users:
    # Get actual products from test set
    actual_products = test_df[test_df['user_id'] == user_id]['product_id'].values
    if len(actual_products) == 0:
        continue
    
    actual_product = actual_products[0]
    
    # Get user features
    user_row = user_features[user_features['customer_id'] == user_id]
    if len(user_row) == 0:
        continue
    
    user_row = user_row.iloc[0]
    
    # Get candidates
    if user_id not in candidates_improved or len(candidates_improved[user_id]) == 0:
        continue
    
    candidates = candidates_improved[user_id][:50]
    
    # Create ranking samples with improved features
    for product_id, cf_score in candidates:
        label = 1 if product_id == actual_product else 0
        
        features = {
            'user_id': user_id,
            'product_id': product_id,
            'label': label,
            'recency': user_row['recency'],
            'frequency': user_row['frequency'],
            'monetary': user_row['monetary'],
            'arpu': user_row['arpu'],
            'usage_7d_data_mb': user_row['usage_7d_data_mb'],
            'churn_score': user_row['churn_score'],
            'cf_score': cf_score,
            # New interaction features
            'segment_id': user_row['segment_id'],
            'freq_x_monetary': user_row['frequency'] * user_row['monetary'],
            'arpu_per_usage': user_row['arpu'] / (user_row['usage_7d_data_mb'] + 1),
            'loyalty_score': user_row['frequency'] * (1 - user_row['churn_score']),
        }
        
        ranking_data_improved.append(features)

ranking_columns = [
    'user_id', 'product_id', 'label', 'recency', 'frequency', 'monetary',
    'arpu', 'usage_7d_data_mb', 'churn_score', 'cf_score', 'segment_id',
    'freq_x_monetary', 'arpu_per_usage', 'loyalty_score'
]
ranking_df_improved = pd.DataFrame(ranking_data_improved, columns=ranking_columns)

print(f"✅ Ranking dataset created")
print(f"   Total samples: {len(ranking_df_improved):,}")

if ranking_df_improved.empty:
    print("⚠️ No ranking samples generated; skipping XGBoost training.")
    train_metrics_improved = {'ndcg@5': 0.0, 'precision@5': 0.0, 'map@5': 0.0}
    test_metrics_improved = {'ndcg@5': 0.0, 'precision@5': 0.0, 'map@5': 0.0}
else:
    print(f"   Positive samples: {ranking_df_improved['label'].sum():,}")
    print(f"   Negative samples: {(ranking_df_improved['label']==0).sum():,}")
    print(f"   Positive ratio: {ranking_df_improved['label'].mean():.4f}")

## 5.3 Train Improved XGBoost Ranker

**Improvements:**
- Better hyperparameters (depth=8, estimators=200)
- Class imbalance handling
- L1/L2 regularization

In [ ]:
if not ranking_df_improved.empty:
    print("\nSTEP 5.3: TRAIN IMPROVED XGBOOST RANKER")
    print("-"*80)
    
    # Prepare features
    feature_cols_improved = [
        'recency', 'frequency', 'monetary', 'arpu',
        'usage_7d_data_mb', 'churn_score', 'cf_score',
        'segment_id', 'freq_x_monetary', 'arpu_per_usage', 'loyalty_score'
    ]
    
    X = ranking_df_improved[feature_cols_improved]
    y = ranking_df_improved['label']
    
    # Split by users
    unique_users = ranking_df_improved['user_id'].unique()
    train_users, test_users = train_test_split(unique_users, test_size=0.2, random_state=42)
    
    train_mask = ranking_df_improved['user_id'].isin(train_users)
    test_mask = ranking_df_improved['user_id'].isin(test_users)
    
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]
    
    train_groups = ranking_df_improved[train_mask].groupby('user_id').size().values
    test_groups = ranking_df_improved[test_mask].groupby('user_id').size().values
    
    print(f"Train: {len(X_train):,} samples ({len(train_groups):,} groups)")
    print(f"Test: {len(X_test):,} samples ({len(test_groups):,} groups)")
    
    # Handle class imbalance
    pos_ratio = y_train.mean()
    if pos_ratio == 0:
        pos_ratio = 1e-6
    scale_pos_weight = (1 - pos_ratio) / pos_ratio
    
    # Improved hyperparameters
    xgb_improved = xgb.XGBRanker(
        objective='rank:pairwise',
        learning_rate=0.05,
        max_depth=8,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )
    
    print(f"\n🚀 Training XGBoost Ranker...")
    print(f"   scale_pos_weight: {scale_pos_weight:.2f}")
    
    xgb_improved.fit(
        X_train, y_train,
        group=train_groups,
        eval_set=[(X_test, y_test)],
        eval_group=[test_groups],
        verbose=False
    )
    
    print("✅ Training complete!")

## 5.4 Evaluate Improved XGBoost

In [ ]:
if not ranking_df_improved.empty and xgb_improved is not None:
    print("\nSTEP 5.4: EVALUATE IMPROVED XGBOOST")
    print("-"*80)
    
    def calculate_ranking_metrics(y_true, y_pred, groups):
        """Calculate NDCG@5, Precision@5, MAP@5"""
        ndcg_scores = []
        precision_scores = []
        map_scores = []
        
        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            
            if group_size < 2:
                start_idx = end_idx
                continue
            
            group_true = y_true.iloc[start_idx:end_idx].values
            group_pred = y_pred[start_idx:end_idx]
            
            # NDCG@5
            k = min(5, group_size)
            try:
                ndcg = ndcg_score([group_true], [group_pred], k=k)
                ndcg_scores.append(ndcg)
            except:
                pass
            
            # Precision@5
            top_k_idx = np.argsort(-group_pred)[:min(5, group_size)]
            precision = group_true[top_k_idx].sum() / min(5, group_size)
            precision_scores.append(precision)
            
            # MAP@5
            sorted_idx = np.argsort(-group_pred)
            ap = 0
            num_relevant = 0
            for i, idx in enumerate(sorted_idx[:5], 1):
                if group_true[idx] == 1:
                    num_relevant += 1
                    ap += num_relevant / i
            if num_relevant > 0:
                ap /= num_relevant
            map_scores.append(ap)
            
            start_idx = end_idx
        
        return {
            'ndcg@5': np.mean(ndcg_scores),
            'precision@5': np.mean(precision_scores),
            'map@5': np.mean(map_scores)
        }
    
    # Evaluate
    y_pred_test = xgb_improved.predict(X_test)
    test_metrics_improved = calculate_ranking_metrics(y_test, y_pred_test, test_groups)
    
    y_pred_train = xgb_improved.predict(X_train)
    train_metrics_improved = calculate_ranking_metrics(y_train, y_pred_train, train_groups)
    
    print(f"\n📊 IMPROVED XGBOOST RESULTS:")
    print(f"\nTRAIN METRICS:")
    print(f"   NDCG@5       : {train_metrics_improved['ndcg@5']:.4f}")
    print(f"   Precision@5  : {train_metrics_improved['precision@5']:.4f}")
    print(f"   MAP@5        : {train_metrics_improved['map@5']:.4f}")
    
    print(f"\nTEST METRICS:")
    print(f"   NDCG@5       : {test_metrics_improved['ndcg@5']:.4f}")
    print(f"   Precision@5  : {test_metrics_improved['precision@5']:.4f}")
    print(f"   MAP@5        : {test_metrics_improved['map@5']:.4f}")
    
    print(f"\n🎯 COMPARISON:")
    print(f"   Original NDCG@5: 0.590")
    print(f"   Improved NDCG@5: {test_metrics_improved['ndcg@5']:.4f}")
    print(f"   Improvement: {((test_metrics_improved['ndcg@5'] - 0.590) / 0.590 * 100):+.1f}%")

---
# PART 6: FINAL PERFORMANCE COMPARISON
---

In [ ]:
print("\n" + "="*80)
print("FINAL PERFORMANCE COMPARISON")
print("="*80)

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Layer': ['K-Means', 'LightFM', 'XGBoost'],
    'Metric': ['Silhouette', 'Precision@5', 'NDCG@5'],
    'Original': [0.252, 0.200, 0.590],
    'Improved': [
        sil_improved,
        test_precision,
        test_metrics_improved.get('ndcg@5', 0) if not ranking_df_improved.empty else 0
    ],
    'Target': [0.60, 0.70, 0.75],
})

comparison['Improvement'] = ((comparison['Improved'] - comparison['Original']) / comparison['Original'] * 100).round(1)
comparison['Target Met'] = comparison.apply(lambda row: '✅' if row['Improved'] >= row['Target'] else '❌', axis=1)

print("\n📊 PERFORMANCE SUMMARY:")
print(comparison.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (ax, layer) in enumerate(zip(axes, ['K-Means', 'LightFM', 'XGBoost'])):
    row = comparison[comparison['Layer'] == layer].iloc[0]
    
    bars = ax.bar(
        ['Original', 'Improved', 'Target'],
        [row['Original'], row['Improved'], row['Target']],
        color=['lightblue', 'lightgreen', 'coral']
    )
    
    ax.set_title(f"{layer} - {row['Metric']}", fontweight='bold')
    ax.set_ylabel('Score')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Key Improvements Summary

In [ ]:
print("\n" + "="*80)
print("KEY IMPROVEMENTS IMPLEMENTED")
print("="*80)
print("""
✅ K-Means Segmentation:
   - Removed correlated features (6 → 4 features)
   - Applied transformations to all skewed features
   - Used RobustScaler for better outlier handling
   - Systematic K optimization (silhouette analysis)
   - Increased n_init (10 → 50) and max_iter (300 → 500)

✅ FixedLightFM Collaborative Filtering:
   - FIXED: Uses product_id (not segment_id)
   - Weighted interactions based on transaction amounts (0.5-1.0)
   - Increased model complexity (50 → 128 components)
   - Chronological train/test split for realistic evaluation
   - More training epochs (30 → 100)
   - Proper cold start handling with graceful fallback
   - WARP loss for ranking optimization

✅ XGBoost Ranking:
   - Added 4 new interaction features
   - Handled class imbalance (scale_pos_weight)
   - Improved hyperparameters (depth, estimators, regularization)
   - Better feature engineering (user × product affinity)
   - Uses FixedLightFM candidates for improved ranking quality
""")

print("\n" + "="*80)
print("🎉 COMPLETE PIPELINE FINISHED!")
print("="*80)

## Save Improved Models

In [ ]:
# Save all improved models
joblib.dump({
    'kmeans': kmeans_improved,
    'scaler': scaler,
    'lightfm': lightfm_improved,
    'xgboost': xgb_improved
}, 'improved_models.pkl')

print("\n✅ Models saved: improved_models.pkl")
print("   - K-Means Segmenter")
print("   - RobustScaler")
print("   - FixedLightFM Collaborative Filtering")
print("   - XGBoost Ranker")
print("✅ Features saved: user_features_improved_seg.csv")